# Notebook 07: Tableau Data Preparation

**Objective:** Export data and predictions for Tableau dashboards

**Outputs:**
- `results/tableau_predictions.csv`: Sampled predictions vs actuals
- `results/tableau_hourly_stats.csv`: Aggregated hourly metrics
- `results/tableau_location_stats.csv`: Aggregated location metrics

---

## 1. Setup

In [5]:
!pip install -q pyspark==3.4.0 pyarrow pandas
print("✓ Libraries installed")


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
✓ Libraries installed


In [6]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.ml.regression import RandomForestRegressionModel
from pyspark.sql.functions import col, avg, count, hour, month, dayofweek

# Create Spark Session
spark = SparkSession.builder \
    .appName("NYC_Taxi_Tableau_Export") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print(f"✓ Spark {spark.version}")

✓ Spark 3.4.0


## 2. Load Data & Model

In [7]:
# Load Gold Data
df_gold = spark.read.parquet("data/gold/nyc_taxi_features")
print(f"✓ Data loaded: {df_gold.count():,} records")

# Load Best Tuned Model (Random Forest)
try:
    model = RandomForestRegressionModel.load("models/tuned/random_forest")
    print("✓ Model loaded: Random Forest (Tuned)")
except:
    print("⚠ Tuned model not found. Using baseline Random Forest...")
    try:
        model = RandomForestRegressionModel.load("models/random_forest")
        print("✓ Model loaded: Random Forest (Baseline)")
    except:
        print("❌ No models found. Please run Notebook 04 or 05 first.")
        model = None

✓ Data loaded: 8,772,267 records
✓ Model loaded: Random Forest (Tuned)


## 3. Generate Predictions Export

In [ ]:
if model:
    # Generate predictions
    predictions = model.transform(df_gold)
    
    # Select columns for Tableau (only columns that exist in Gold layer)
    tableau_cols = [
        "label", "prediction", 
        "trip_distance", "passenger_count", 
        "pickup_hour",
        "is_rush_hour", "is_weekend", "is_airport_trip",
        "VendorID"
    ]
    
    # Sample 1% of data for Tableau (to keep CSV size manageable)
    # 1% of ~8M records = ~80k records (perfect for Tableau Public)
    df_export = predictions.select(tableau_cols).sample(fraction=0.01, seed=42).toPandas()
    
    # Rename for clarity
    df_export = df_export.rename(columns={
        "label": "Actual_Fare",
        "prediction": "Predicted_Fare"
    })
    
    # Save
    df_export.to_csv("results/tableau_predictions.csv", index=False)
    print(f"✓ Exported {len(df_export):,} prediction records to 'results/tableau_predictions.csv'")

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `pickup_day_of_week` cannot be resolved. Did you mean one of the following? [`pickup_hour`, `tpep_pickup_datetime`, `data_year`, `trip_distance`, `is_airport_trip`].;
'Project [label#92, prediction#164, trip_distance#93, passenger_count#94, pickup_hour#95, 'pickup_day_of_week, 'pickup_month, is_rush_hour#96, is_weekend#97, is_airport_trip#98, 'RatecodeID, 'PULocationID, 'DOLocationID]
+- Project [features#91, label#92, trip_distance#93, passenger_count#94, pickup_hour#95, is_rush_hour#96, is_weekend#97, is_airport_trip#98, VendorID#99, tpep_pickup_datetime#100, data_year#101, data_month#102, UDF(features#91) AS prediction#164]
   +- Relation [features#91,label#92,trip_distance#93,passenger_count#94,pickup_hour#95,is_rush_hour#96,is_weekend#97,is_airport_trip#98,VendorID#99,tpep_pickup_datetime#100,data_year#101,data_month#102] parquet


## 4. Aggregate Business Insights

In [ ]:
# 1. Hourly Stats
hourly_stats = df_gold.groupBy("pickup_hour").agg(
    avg("label").alias("avg_fare"),
    avg("trip_distance").alias("avg_distance"),
    count("*").alias("trip_count")
).orderBy("pickup_hour").toPandas()

hourly_stats.to_csv("results/tableau_hourly_stats.csv", index=False)
print("✓ Exported hourly stats")

# 2. Rush Hour vs Non-Rush Hour Stats
rush_hour_stats = df_gold.groupBy("is_rush_hour").agg(
    avg("label").alias("avg_fare"),
    avg("trip_distance").alias("avg_distance"),
    count("*").alias("trip_count")
).toPandas()

rush_hour_stats.to_csv("results/tableau_rush_hour_stats.csv", index=False)
print("✓ Exported rush hour stats")

# 3. Weekend vs Weekday Stats
weekend_stats = df_gold.groupBy("is_weekend").agg(
    avg("label").alias("avg_fare"),
    avg("trip_distance").alias("avg_distance"),
    count("*").alias("trip_count")
).toPandas()

weekend_stats.to_csv("results/tableau_weekend_stats.csv", index=False)
print("✓ Exported weekend stats")

# 4. Airport Trip Stats
airport_stats = df_gold.groupBy("is_airport_trip").agg(
    avg("label").alias("avg_fare"),
    avg("trip_distance").alias("avg_distance"),
    count("*").alias("trip_count")
).toPandas()

airport_stats.to_csv("results/tableau_airport_stats.csv", index=False)
print("✓ Exported airport trip stats")

## Summary

**Generated Files for Tableau:**
1. `results/tableau_predictions.csv` - For Model Performance Dashboard (Actual vs Predicted)
2. `results/tableau_hourly_stats.csv` - For Business Insights (Hourly patterns)
3. `results/tableau_rush_hour_stats.csv` - For Business Insights (Rush hour analysis)
4. `results/tableau_weekend_stats.csv` - For Business Insights (Weekend vs weekday)
5. `results/tableau_airport_stats.csv` - For Business Insights (Airport trip analysis)

**Next Steps:**
1. Open Tableau Public/Desktop
2. Connect to these CSV files
3. Build your dashboards

**Note:** The Gold layer contains only selected columns. If you need additional columns (like pickup_day_of_week, pickup_month, RatecodeID, PULocationID, DOLocationID), you would need to load them from the Silver layer and join with predictions.